## Freeze-and-Train Deep Ensemble for Mortality Prediction

### Import packages

In [1]:
import tensorflow as tf
import numpy as np
import os as os
tfkl = tf.keras.layers
import random

### Import functions

In [2]:
import training_functions_freeze_uncertainty as training_functions
import importlib

importlib.reload(training_functions)

<module 'training_functions_freeze_uncertainty' from '/Users/paigepark/Desktop/repos/deep-mort/code/training_functions_freeze_uncertainty.py'>

### Import data

#### Country data

In [ ]:
country_training = np.loadtxt('../../data/country_training.txt')
country_test = np.loadtxt('../../data/country_test.txt')

In [4]:
country_training.shape

(359594, 5)

#### All Country Model (Freeze-and-Train Ensemble)

In [6]:
# prep data
country_train_prepped = training_functions.prep_data(country_training, mode="train", changeratetolog=True)
country_test_prepped = training_functions.prep_data(country_test, mode="test", changeratetolog=True)

In [7]:
# get the proper geography input dimension for model set up 
unique_vals = tf.unique(country_training[:, 0]).y
country_geo_dim = np.array(tf.size(unique_vals)).item()
country_geo_dim = country_geo_dim + 50

In [8]:
# Prepare input features once (shared across all iterations)
training_input_features = (
    tf.convert_to_tensor((country_training[:, 2] - 1959) / 60, dtype=tf.float32),
    tf.convert_to_tensor(country_training[:, 3], dtype=tf.float32),
    tf.convert_to_tensor(country_training[:, 0], dtype=tf.float32),
    tf.convert_to_tensor(country_training[:, 1], dtype=tf.float32),
)

test_input_features = (
    tf.convert_to_tensor((country_test[:, 2] - 1959) / 60, dtype=tf.float32),
    tf.convert_to_tensor(country_test[:, 3], dtype=tf.float32),
    tf.convert_to_tensor(country_test[:, 0], dtype=tf.float32),
    tf.convert_to_tensor(country_test[:, 1], dtype=tf.float32),
)

inputs_train = np.delete(country_training, 4, axis=1)
inputs_test = np.delete(country_test, 4, axis=1)

# Storage for ensemble combination
M = 5
train_mus = []
train_vars = []
test_mus = []
test_vars = []

In [ ]:
# train each ensemble member with different seeds, saving predictions and models per member
for i in range(1, M + 1):
    # Set reproducible seeds per iteration
    np.random.seed(i)
    tf.random.set_seed(i)
    random.seed(i)
    os.environ['PYTHONHASHSEED'] = str(i)

    model_country, loss_info_country = training_functions.run_freeze_ensemble_model(
        country_train_prepped, country_test_prepped, country_geo_dim,
        epochs_mean=20, epochs_var=10, steps_per_epoch=1405, lograte=True
    )

    # Get raw predictions: shape (N, 2) = [mu, raw_variance]
    raw_train_preds = model_country.predict(training_input_features)
    raw_test_preds = model_country.predict(test_input_features)

    # Split into mu and variance (applying softplus + floor to raw variance)
    train_mu = raw_train_preds[:, 0:1]
    train_var = np.log(1.0 + np.exp(raw_train_preds[:, 1:2])) + 1e-6

    test_mu = raw_test_preds[:, 0:1]
    test_var = np.log(1.0 + np.exp(raw_test_preds[:, 1:2])) + 1e-6

    # Store for ensemble combination
    train_mus.append(train_mu)
    train_vars.append(train_var)
    test_mus.append(test_mu)
    test_vars.append(test_var)

    # Save individual member predictions: [geo, gender, year, age, mu, variance]
    training_predictions = np.column_stack((inputs_train, train_mu, train_var))
    test_predictions = np.column_stack((inputs_test, test_mu, test_var))

    model_country.save(f"../../models/dl_country_freeze_ensemble_{i}.keras")
    np.savetxt(f"../../data/dl_country_freeze_fitted_{i}.txt", training_predictions)
    np.savetxt(f"../../data/dl_country_freeze_forecast_{i}.txt", test_predictions)

    print(f"Ensemble member {i} complete | val NLL: {loss_info_country:.6f}")

In [10]:
# combine ensemble members to get overall predictions and uncertainty decomposition
train_mus = np.array(train_mus)    # (M, N, 1)
train_vars = np.array(train_vars)  # (M, N, 1)
test_mus = np.array(test_mus)      # (M, N, 1)
test_vars = np.array(test_vars)    # (M, N, 1)

# Ensemble mean: average of individual means
ensemble_train_mu = np.mean(train_mus, axis=0)
ensemble_test_mu = np.mean(test_mus, axis=0)

# Ensemble variance: (1/M) * sum(sigma_m^2 + mu_m^2) - mu*^2
ensemble_train_var = np.mean(train_vars + train_mus**2, axis=0) - ensemble_train_mu**2
ensemble_test_var = np.mean(test_vars + test_mus**2, axis=0) - ensemble_test_mu**2

# Decompose into aleatoric (data noise) and epistemic (model disagreement)
aleatoric_train = np.mean(train_vars, axis=0)
epistemic_train = np.mean(train_mus**2, axis=0) - ensemble_train_mu**2

aleatoric_test = np.mean(test_vars, axis=0)
epistemic_test = np.mean(test_mus**2, axis=0) - ensemble_test_mu**2

# 95% prediction intervals
z = 1.96
ensemble_train_std = np.sqrt(ensemble_train_var)
ensemble_test_std = np.sqrt(ensemble_test_var)

train_lower = ensemble_train_mu - z * ensemble_train_std
train_upper = ensemble_train_mu + z * ensemble_train_std
test_lower = ensemble_test_mu - z * ensemble_test_std
test_upper = ensemble_test_mu + z * ensemble_test_std

In [ ]:
# Save: [geo, gender, year, age, mu, total_var, aleatoric_var, epistemic_var, lower_95, upper_95]
ensemble_train_output = np.column_stack((
    inputs_train, ensemble_train_mu, ensemble_train_var,
    aleatoric_train, epistemic_train, train_lower, train_upper
))
ensemble_test_output = np.column_stack((
    inputs_test, ensemble_test_mu, ensemble_test_var,
    aleatoric_test, epistemic_test, test_lower, test_upper
))

np.savetxt("../../data/dl_country_freeze_ensemble_fitted.txt", ensemble_train_output)
np.savetxt("../../data/dl_country_freeze_ensemble_forecast.txt", ensemble_test_output)

print("\nFreeze-and-train ensemble combination complete.")
print(f"Output columns: geo, gender, year, age, mu, total_var, aleatoric_var, epistemic_var, lower_95, upper_95")
print(f"Train shape: {ensemble_train_output.shape}")
print(f"Test shape:  {ensemble_test_output.shape}")